## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
#include <iostream>
#include <vector>
#include <algorithm>
#include <cstdint>
#include <utility>

using namespace std;

// ============================================================================
// 基本操作指令
// ============================================================================
struct Operation {
    enum Type : int { Swap = 0, Xor = 1, Add = 2 };

    Type type;
    int param;

    static constexpr Operation makeSwap() noexcept { return {Swap, 0}; }
    static constexpr Operation makeXor(int val) noexcept { return {Xor, val}; }
    static constexpr Operation makeAdd(int val) noexcept { return {Add, val}; }
};

// ============================================================================
// 前缀求解器 —— 处理低比特排列
// ============================================================================
namespace PrefixSolver {

// 检查序列是否为有效的 [0, maxVal) 排列
[[nodiscard]] bool isValidPermutation(const vector<int>& seq, int maxVal) {
    vector<bool> seen(maxVal, false);
    for (int x : seq) {
        if (x < 0 || x >= maxVal || seen[x]) return false;
        seen[x] = true;
    }
    return true;
}

// 插入非零异或操作
void appendXor(vector<Operation>& ops, int val) {
    if (val != 0) ops.push_back(Operation::makeXor(val));
}

// 插入非零加法操作
void appendAdd(vector<Operation>& ops, int val) {
    if (val != 0) ops.push_back(Operation::makeAdd(val));
}

// 合并连续异或，并移除空操作
void compressXorOps(vector<Operation>& ops) {
    vector<Operation> compact;
    for (const auto& op : ops) {
        if (op.type == Operation::Xor && !compact.empty() && compact.back().type == Operation::Xor) {
            int combined = compact.back().param ^ op.param;
            compact.pop_back();
            if (combined != 0) compact.push_back(Operation::makeXor(combined));
        } else {
            if (op.type == Operation::Swap || op.param != 0)
                compact.push_back(op);
        }
    }
    ops = std::move(compact);
}

// 递归分解排列为操作序列
[[nodiscard]] bool decompose(const vector<int>& perm, int limit, vector<Operation>& outOps) {
    outOps.clear();
    if (!isValidPermutation(perm, limit)) return false;
    if (limit == 1) return true;

    int half = limit >> 1;
    vector<int> evens(half), odds(half);
    for (int i = 0; i < half; ++i) {
        evens[i] = perm[i << 1] >> 1;
        odds[i]  = perm[(i << 1) | 1] >> 1;
    }

    vector<Operation> leftOps, rightOps;
    if (!decompose(evens, half, leftOps) || !decompose(odds, half, rightOps))
        return false;

    // 最低位校正
    if (perm[0] & 1) {
        if (limit == 2)
            appendAdd(outOps, 1);
        else
            appendXor(outOps, 1);
    }

    int leftAcc = 0, rightAcc = 0;

    for (const auto& op : leftOps) {
        if (op.type == Operation::Add) {
            appendXor(outOps, 1);
            appendAdd(outOps, 1);
        } else {
            appendXor(outOps, op.param << 1);
            leftAcc ^= (op.param << 1);
        }
    }
    appendXor(outOps, leftAcc);

    for (const auto& op : rightOps) {
        if (op.type == Operation::Add) {
            appendAdd(outOps, 1);
            appendXor(outOps, 1);
        } else {
            appendXor(outOps, op.param << 1);
            rightAcc ^= (op.param << 1);
        }
    }

    // 验证一致性
    if ((leftAcc & half) != (rightAcc & half)) return false;
    if (leftAcc >= half) leftAcc -= half;
    if (rightAcc >= half) rightAcc -= half;
    if (leftAcc != rightAcc) return false;

    compressXorOps(outOps);
    return true;
}

} // namespace PrefixSolver

// ============================================================================
// 状态引擎 —— 执行排列复原
// ============================================================================
class StateEngine {
public:
    StateEngine(int totalSize, int anchorU, int anchorV, vector<int> initialState)
        : m_size(totalSize)
        , m_anchorU(anchorU)
        , m_anchorV(anchorV)
        , m_state(std::move(initialState))
    {
        int delta = (m_anchorU - m_anchorV + m_size) % m_size;
        m_stride = (delta == 0) ? m_size : (delta & -delta); // 最低非零位
    }

    // 执行算法，成功返回 true，可从 getOperations() 获取操作序列
    [[nodiscard]] bool run() {
        if (!isValidInitialState()) return false;

        fixLowBits();                 // 步骤1：修正低 stride 位
        sortGroupsAndSwap();          // 步骤2：按同余类排序并交换到目标位置
        return verifyFinalState();    // 步骤3：最终检查
    }

    [[nodiscard]] const vector<Operation>& getOperations() const { return m_operations; }

private:
    int m_size;                     // 总元素数
    int m_anchorU, m_anchorV;       // 锚点
    int m_stride;                   // 步长
    vector<int> m_state;            // 当前排列
    vector<Operation> m_operations; // 记录已执行的操作

    // ---------- 工具函数 ----------
    bool isValidInitialState() const {
        vector<bool> used(m_size, false);
        for (int x : m_state) {
            if (x < 0 || x >= m_size || used[x]) return false;
            used[x] = true;
        }
        return true;
    }

    void applyOperation(const Operation& op) {
        if ((op.type == Operation::Xor || op.type == Operation::Add) && op.param == 0)
            return; // 跳过空操作

        m_operations.push_back(op);

        if (op.type == Operation::Swap) {
            for (int& val : m_state) {
                if (val == m_anchorU)      val = m_anchorV;
                else if (val == m_anchorV) val = m_anchorU;
            }
        } else if (op.type == Operation::Xor) {
            for (int& val : m_state) val ^= op.param;
        } else { // Add
            for (int& val : m_state) val = (val + op.param) % m_size;
        }
    }

    void shiftBy(int offset) {
        offset %= m_size;
        if (offset < 0) offset += m_size;
        if (offset != 0) applyOperation(Operation::makeAdd(offset));
    }

    void xorBy(int mask) {
        if (mask != 0) applyOperation(Operation::makeXor(mask));
    }

    // 计算连接“桥接点”，用于异类块间的交换
    [[nodiscard]] pair<int, int> computeBridge(int u, int v) const {
        int dist = (v - u + m_size - m_stride + m_size) % m_size;
        int pt1 = 0, pt2 = 0;

        for (int chunk = m_size >> 1; chunk >= (m_stride << 1); chunk >>= 1) {
            if (dist >= chunk) {
                dist -= chunk;
                pt2 += chunk >> 1;
            } else {
                pt1 += chunk >> 1;
            }
        }

        int remainder = u & (m_stride - 1);
        pt1 += (m_size >> 1) + remainder;
        pt2 += remainder;

        return {pt1, pt2};
    }

    // ---------- 核心算法 ----------
    void fixLowBits() {
        if (m_stride <= 1) return;

        vector<int> lowBits(m_stride);
        for (int i = 0; i < m_stride; ++i)
            lowBits[i] = m_state[i] & (m_stride - 1);

        vector<Operation> prefixOps;
        if (!PrefixSolver::decompose(lowBits, m_stride, prefixOps))
            return; // 合法性已在 run() 入口验证，此处不应失败

        for (const auto& op : prefixOps)
            applyOperation(op);
    }

    void sortGroupsAndSwap() {
        // 对每个同余类（模 stride）检查并纠正位置
        for (int rem = 0; rem < m_stride; ++rem) {
            // 收集该组实际值并排序
            vector<int> group;
            for (int pos = rem; pos < m_size; pos += m_stride)
                group.push_back(m_state[pos]);
            sort(group.begin(), group.end());

            // 验证排序后是否恰好为 {rem, rem+stride, ...}
            int index = 0;
            for (int pos = rem; pos < m_size; pos += m_stride, ++index) {
                if (group[index] != pos) return; // 无解，调用者会返回失败
            }

            // 执行交换，将每个元素放到正确位置
            for (int pos = rem; pos < m_size; pos += m_stride) {
                while (m_state[pos] != pos)
                    swapArbitrary(pos, m_state[pos]);
            }
        }
    }

    bool verifyFinalState() const {
        for (int i = 0; i < m_size; ++i)
            if (m_state[i] != i) return false;
        return true;
    }

    // 任意位置交换（通过递归归约到锚点附近）
    void swapArbitrary(int u, int v) {
        if (u == v) return;

        int blockU = (u / m_stride) & 1;
        int blockV = (v / m_stride) & 1;

        // 同侧块：引入中间点，转为三个异侧交换
        if (blockU == blockV) {
            int mid = (blockU == 0) ? ((u & (m_stride - 1)) + m_stride)
                                    : (u & (m_stride - 1));
            swapArbitrary(u, mid);
            swapArbitrary(v, mid);
            swapArbitrary(u, mid);
            return;
        }

        // 异侧块：通过全局移位/异或操作完成交换
        auto targetPair = computeBridge(m_anchorU, m_anchorV);
        auto srcPair    = computeBridge(u, v);

        shiftBy(srcPair.first - u);
        xorBy(srcPair.first ^ targetPair.first);
        shiftBy(m_anchorU - targetPair.first);

        applyOperation(Operation::makeSwap());

        shiftBy(targetPair.first - m_anchorU);
        xorBy(srcPair.first ^ targetPair.first);
        shiftBy(u - srcPair.first);
    }
};

// ============================================================================
// 主程序入口
// ============================================================================
int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(nullptr);

    int m, a, b;
    if (!(cin >> m >> a >> b)) return 0;

    vector<int> initial(m);
    for (int i = 0; i < m; ++i)
        cin >> initial[i];

    StateEngine engine(m, a, b, std::move(initial));

    if (!engine.run()) {
        cout << "-1\n";
    } else {
        const auto& ops = engine.getOperations();
        cout << ops.size() << '\n';
        for (const auto& op : ops) {
            if (op.type == Operation::Swap) {
                cout << "0\n";
            } else {
                cout << static_cast<int>(op.type) << ' ' << op.param << '\n';
            }
        }
    }

    return 0;
}

## B 长跑

In [ ]:
## add your code here#include <stdio.h>
#include <stdlib.h>

#define INF 1000000000 // 定义一个足够大的数表示“不可达”状态

// 定义补给站结构体
typedef struct {
    int pos;  // 补给站位置
    int cost; // 补给站花费
} Station;

// 用于qsort排序的比较函数，按位置从小到大排序
int cmp(const void *a, const void *b) {
    Station *sa = (Station *)a;
    Station *sb = (Station *)b;
    return sa->pos - sb->pos;
}

int main() {
    int N, L, Maxn, S;
    
    // 题目包含多组数据，使用 while 读取直到文件结束
    while (scanf("%d %d %d %d", &N, &L, &Maxn, &S) == 4) {
        Station stations[2005];
        
        // 读取所有补给站的信息
        for (int i = 0; i < N; i++) {
            scanf("%d %d", &stations[i].pos, &stations[i].cost);
        }
        
        // 按照补给站所在位置从左到右排序
        qsort(stations, N, sizeof(Station), cmp);
        
        // dp[i] 表示到达并使用第 i 个补给站所需的最小花费
        int dp[2005];
        for (int i = 0; i < N; i++) {
            dp[i] = INF; // 初始化为不可达
        }
        
        // 动态规划计算每个补给站的最小花费
        for (int i = 0; i < N; i++) {
            // 1. 如果起点可以直接跑到当前补给站 (距离 <= Maxn)
            if (stations[i].pos <= Maxn) {
                dp[i] = stations[i].cost;
            }
            
            // 2. 如果是从前面的某个补给站 j 跑到当前补给站 i
            for (int j = 0; j < i; j++) {
                // 如果补给站 j 是可达的，并且从 j 到 i 的距离小明能跑得到
                if (dp[j] != INF && stations[i].pos - stations[j].pos <= Maxn) {
                    // 更新最小花费
                    if (dp[j] + stations[i].cost < dp[i]) {
                        dp[i] = dp[j] + stations[i].cost;
                    }
                }
            }
        }
        
        int min_total_cost = INF;
        
        // 检查直接从起点跑到终点的情况
        if (L <= Maxn) {
            min_total_cost = 0; // 一次补给都不需要，花费为0
        }
        
        // 检查从各个补给站跑到终点的情况
        for (int i = 0; i < N; i++) {
            // 补给站必须在终点之前(或正好在终点)，且从该补给站能跑到终点
            if (stations[i].pos <= L && L - stations[i].pos <= Maxn) {
                if (dp[i] < min_total_cost) {
                    min_total_cost = dp[i];
                }
            }
        }
        
        // 判断最小花费是否小于等于小明手上的硬币 S
        if (min_total_cost <= S) {
            printf("Yes\n");
        } else {
            printf("No\n");
        }
    }
    
    return 0;
}

## C 最长回文

In [ ]:
## add your code here
#include <stdio.h>
#include <string.h>
#include <stdlib.h>

#define MAX_N 200005 // 字符串最大长度安全余量

char A[MAX_N], B[MAX_N];
char a[MAX_N * 2], b[MAX_N * 2];
int pa[MAX_N * 2], pb[MAX_N * 2];

int min_val(int x, int y) { return x < y ? x : y; }
int max_val(int x, int y) { return x > y ? x : y; }

// 标准Manacher处理
void manacher(char *s, int n, char *t, int *p) {
    t[0] = '$';
    t[1] = '#';
    for (int i = 0; i < n; i++) {
        t[2 * i + 2] = s[i];
        t[2 * i + 3] = '#';
    }
    t[2 * n + 2] = '\0';
    
    int mx = 0, id = 0;
    int len = 2 * n + 2;
    for (int i = 1; i < len; i++) {
        p[i] = mx > i ? min_val(p[2 * id - i], mx - i) : 1;
        while (t[i + p[i]] == t[i - p[i]]) {
            p[i]++;
        }
        if (i + p[i] > mx) {
            mx = i + p[i];
            id = i;
        }
    }
}

int main() {
    int n;
    // 处理多组数据 (鲁棒性写法)
    while (scanf("%d", &n) == 1) {
        scanf("%s", A);
        scanf("%s", B);
        
        if (n == 0) {
            printf("0\n");
            continue;
        }

        // 清空半径数组
        memset(pa, 0, sizeof(int) * (2 * n + 5));
        memset(pb, 0, sizeof(int) * (2 * n + 5));
        
        // 跑两边Manacher进行预处理
        manacher(A, n, a, pa);
        manacher(B, n, b, pb);
        
        int ans = 1; // 单个字符至少能构成长度为1的回文串
        
        // 枚举每一个可能的回文中心 (包括字符和中间的#号)
        // 遍历所有扩充后的串的可能核心位置
        for (int i = 2; i <= 2 * n + 2; i++) {
            // 中心为 i 时，A 提供 i，B 对应的对接位为 i - 2
            // 贪心地从原串A、B各自最大的回文半径直接起步
            int len = max_val(pa[i], pb[i - 2]);
            if (len < 1) len = 1; 
            
            // 如果 A的左翼 继续等于 B的右翼，说明跨字符串的回文串可以继续延展
            while (i - len >= 0 && i - 2 + len < 2 * n + 2 && a[i - len] == b[i - 2 + len]) {
                len++;
            }
            
            // 更新当前能组成的最长回文长度
            ans = max_val(ans, len - 1);
        }
        
        printf("%d\n", ans);
    }
    return 0;
}

## D 优惠券

In [ ]:
## add your code here
#include <stdio.h>
#include <string.h>

#define MAX_M 500005
#define MAX_X 100005

// parent数组用于并查集，快速跳过已经使用的 '?'
int parent[MAX_M];
int cnt[MAX_X];       // 记录优惠券 x 当前持有的数量 (只会是 0 或 1)
int vis[MAX_X];       // 记录优惠券 x 上一次被操作所在的行号
int last_case[MAX_X]; // 时间戳：避免 memset 造成超时，极速清空数组
int case_id = 0;

// 并查集查找：寻找 i 之后(含i)第一个可用的 '?'
int get_parent(int i) {
    int root = i;
    while (root != parent[root]) {
        root = parent[root];
    }
    // 路径压缩，保证下一次查找是 O(1)
    int curr = i;
    while (curr != root) {
        int nxt = parent[curr];
        parent[curr] = root;
        curr = nxt;
    }
    return root;
}

// O(1) 极速初始化优惠券的状态
void init_x(int x) {
    if (last_case[x] != case_id) {
        last_case[x] = case_id;
        cnt[x] = 0;
        vis[x] = 0;
    }
}

int main() {
    int m;
    
    // 稳健读取多组测试数据
    while (scanf("%d", &m) == 1) {
        case_id++; 
        int error_line = -1;
        
        // 初始化并查集
        for (int i = 1; i <= m + 1; ++i) {
            parent[i] = i;
        }
        
        for (int i = 1; i <= m; ++i) {
            char str[105];
            // 放弃 %c，改用 %s 读字符串，天然过滤空格回车，且能完整读取异常全角字符
            if (scanf("%s", str) != 1) break;
            
            char type = 0;
            // 兼容半角 '?' 和 你误输入的 全角 '？'
            if (str[0] == '?' || strstr(str, "？") != NULL) {
                type = '?';
            } else if (str[0] == 'I') {
                type = 'I';
            } else if (str[0] == 'O') {
                type = 'O';
            } else {
                type = '?'; // 防御性编程：遇到不可识别垃圾字符，当做占位符处理
            }
            
            if (type == '?') {
                // 如果是 '?'，当前可用，parent 指向自己，无需多做处理
            } 
            else if (type == 'I' || type == 'O') {
                int x;
                scanf("%d", &x); // 只有明确是 I 或 O 时，才去读数字，绝不污染缓冲区
                
                // 既然当前行是确定的 I 或 O，它就不能作为万能牌，指向下一行
                parent[i] = i + 1; 
                
                if (error_line == -1) {
                    init_x(x); // 确保干净的状态
                    
                    if (type == 'I') {
                        if (cnt[x] == 1) {
                            // 发生冲突：已经买过 1 张，不能重复购买！
                            int q_idx = get_parent(vis[x] + 1);
                            if (q_idx < i) { // 必须是在当前行之前找到的 '?' 才有意义
                                parent[q_idx] = q_idx + 1; // 消耗掉这个 '?'
                            } else {
                                error_line = i; // 找不到，死局，记录报错行
                            }
                        } else {
                            cnt[x] = 1; // 正常购买
                        }
                        vis[x] = i; // 记录 x 最后的活动时间
                    } 
                    else if (type == 'O') {
                        if (cnt[x] == 0) {
                            // 发生冲突：手里根本没有这张券，却要使用它！
                            int q_idx = get_parent(vis[x] + 1);
                            if (q_idx < i) {
                                parent[q_idx] = q_idx + 1; // 消耗掉这个 '?'
                            } else {
                                error_line = i; // 找不到，死局，报错
                            }
                        } else {
                            cnt[x] = 0; // 正常消耗
                        }
                        vis[x] = i; // 记录 x 最后的活动时间
                    }
                }
            }
        }
        
        // 这一组数据完全读取完毕后统一输出
        if (error_line != -1) {
            printf("%d\n", error_line);
        } else {
            printf("-1\n");
        }
    }
    
    return 0;
}

## E 任意点

In [ ]:
#include <stdio.h>

#define MAX_N 105

// 并查集的 parent 数组
int parent[MAX_N];

// 查找根节点，并进行路径压缩优化
int find(int i) {
    if (parent[i] == i) {
        return i;
    }
    // 路径压缩：直接将当前节点指向根节点，加速后续查找
    return parent[i] = find(parent[i]);
}

// 合并两个集合
void union_sets(int i, int j, int *count) {
    int root_i = find(i);
    int root_j = find(j);
    // 如果两个点不在同一个集合，则合并它们
    if (root_i != root_j) {
        parent[root_i] = root_j;
        (*count)--; // 合并一次，总的连通分量数量减少 1
    }
}

int main() {
    int n;
    
    // 读取点数，使用 while 处理可能的连续多组测试数据
    while (scanf("%d", &n) != EOF) {
        int x[MAX_N], y[MAX_N];
        
        // 读取每个点的坐标，并初始化并查集
        for (int i = 0; i < n; i++) {
            scanf("%d %d", &x[i], &y[i]);
            parent[i] = i; // 初始时，每个点的父节点是自己
        }
        
        int count = n; // 初始连通分量数量为点数 n
        
        // 遍历所有点对，检查是否可以相连
        for (int i = 0; i < n; i++) {
            for (int j = i + 1; j < n; j++) {
                // 如果横坐标相同，或纵坐标相同，说明它们可以直接互达
                if (x[i] == x[j] || y[i] == y[j]) {
                    union_sets(i, j, &count);
                }
            }
        }
        
        // 需要添加的最少点数 = 最终的连通分量数量 - 1
        printf("%d\n", count - 1);
    }
    
    return 0;
}

## F 通配符匹配

In [ ]:
#include <stdio.h>
#include <string.h>

#define MAXL 100005

// 使用 unsigned long long 自动溢出实现模 2^64 哈希，速度最快
typedef unsigned long long ull;
ull h[MAXL], pw[MAXL];
const ull base = 131; // 常用哈希种子

char pattern[MAXL];
char filename[MAXL];
char *F[15];     // 存储切分后的固定字符串段
char W[15];      // 存储通配符类型
int F_len[15];   // 存储固定段长度
ull F_hash[15];  // 存储固定段的哈希值
int K;           // 通配符总数
signed char memo[15][MAXL];
int f_len;

// 预处理哈希幂次
void init_pow() {
    pw[0] = 1;
    for (int i = 1; i < MAXL; i++) pw[i] = pw[i - 1] * base;
}

// 预处理文件名的前缀哈希
void init_filename_hash() {
    h[0] = 0;
    for (int i = 1; i <= f_len; i++) {
        h[i] = h[i - 1] * base + filename[i - 1];
    }
}

// O(1) 获取文件名子串的哈希值
ull get_hash(int l, int len) {
    return h[l + len] - h[l] * pw[len];
}

// 计算固定段的哈希值
ull compute_str_hash(char *s, int len) {
    ull res = 0;
    for (int i = 0; i < len; i++) res = res * base + s[i];
    return res;
}

// 核心 DFS 逻辑
int dfs(int idx, int pos) {
    if (idx > K) return pos == f_len; // 必须匹配到文件名末尾
    if (memo[idx][pos] != -1) return memo[idx][pos];

    int res = 0;
    if (idx == 0) {
        // 处理第一个固定段（前缀）
        if (pos + F_len[0] <= f_len && get_hash(pos, F_len[0]) == F_hash[0]) {
            res = dfs(1, pos + F_len[0]);
        }
    } else {
        if (W[idx] == '?') {
            // '?' 必须跳过且仅跳过 1 个字符，然后匹配随后的固定段
            if (pos < f_len) {
                int next_pos = pos + 1;
                if (next_pos + F_len[idx] <= f_len && get_hash(next_pos, F_len[idx]) == F_hash[idx]) {
                    res = dfs(idx + 1, next_pos + F_len[idx]);
                }
            }
        } else { // '*' 通配符
            // 情况 1: '*' 停止匹配（匹配 0 个或已匹配结束），尝试匹配 F[idx]
            if (pos + F_len[idx] <= f_len && get_hash(pos, F_len[idx]) == F_hash[idx]) {
                if (dfs(idx + 1, pos + F_len[idx])) res = 1;
            }
            // 情况 2: '*' 继续匹配至少 1 个字符
            if (!res && pos < f_len) {
                if (dfs(idx, pos + 1)) res = 1;
            }
        }
    }
    return memo[idx][pos] = (signed char)res;
}

int main() {
    init_pow();
    if (scanf("%s", pattern) != 1) return 0;
    int n;
    if (scanf("%d", &n) != 1) return 0;

    // 预处理模式串：切分为 F0 W1 F1 W2 F2 ...
    K = 0;
    char *p = pattern;
    F[0] = p;
    int cur_len = 0;
    while (*p) {
        if (*p == '*' || *p == '?') {
            char type = *p;
            *p = '\0';
            F_len[K] = cur_len;
            F_hash[K] = compute_str_hash(F[K], cur_len);
            K++;
            W[K] = type;
            F[K] = p + 1;
            cur_len = 0;
        } else {
            cur_len++;
        }
        p++;
    }
    F_len[K] = cur_len;
    F_hash[K] = compute_str_hash(F[K], cur_len);

    while (n--) {
        if (scanf("%s", filename) != 1) break;
        f_len = (int)strlen(filename);
        init_filename_hash();
        
        // 初始化备忘录，仅清理当前文件名长度相关的部分以节省时间
        for (int i = 0; i <= K; i++) {
            for (int j = 0; j <= f_len; j++) memo[i][j] = -1;
        }

        if (dfs(0, 0)) printf("YES\n");
        else printf("NO\n");
    }
    return 0;
}

## G 汉诺塔

In [ ]:
#include <stdio.h>
#include <string.h>

typedef long long ll;

int main() {
    int n;
    char p[6][3];
    ll f[35][3];
    int to[35][3];

    // 读取输入
    if (scanf("%d", &n) != 1) return 0;
    for (int i = 0; i < 6; i++) {
        scanf("%s", p[i]);
    }

    // 初始化 n=1 的情况
    for (int i = 0; i < 3; i++) {
        char start_peg = 'A' + i;
        for (int j = 0; j < 6; j++) {
            if (p[j][0] == start_peg) {
                f[1][i] = 1;
                to[1][i] = p[j][1] - 'A';
                break;
            }
        }
    }

    // 动态规划计算 n > 1 的情况
    for (int k = 2; k <= n; k++) {
        for (int i = 0; i < 3; i++) {
            int mid = to[k - 1][i];
            int other = 3 - i - mid;

            // 逻辑：先移走 n-1 个盘子到 mid，再移第 n 个到 other
            // 然后看 n-1 个盘子从 mid 出发后的去向
            if (to[k - 1][mid] == other) {
                // n-1 塔直接落到了第 n 个盘子上面
                f[k][i] = f[k - 1][i] + 1 + f[k - 1][mid];
                to[k][i] = other;
            } else {
                // n-1 塔回到了原位，第 n 个盘子需要再跳一根柱子，塔再跟过去
                f[k][i] = f[k - 1][i] + 1 + f[k - 1][mid] + 1 + f[k - 1][i];
                to[k][i] = mid;
            }
        }
    }

    // 输出从 A 柱 (0) 开始移动 n 个盘子的步数
    printf("%lld\n", f[n][0]);

    return 0;
}

## H 马步距离

In [ ]:
#include <stdio.h>
#include <stdlib.h>

/**
 * 计算马从 (xp, yp) 到 (xs, ys) 的最短步数
 */
int main() {
    long long xp, yp, xs, ys;
    
    // 读取输入坐标
    if (scanf("%lld %lld %lld %lld", &xp, &yp, &xs, &ys) != 4) {
        return 0;
    }

    // 计算相对坐标的绝对值
    long long dx = llabs(xp - xs);
    long long dy = llabs(yp - ys);

    // 标准化，令 x 为较大值，y 为较小值，保证 x >= y >= 0
    long long x = (dx > dy) ? dx : dy;
    long long y = (dx > dy) ? dy : dx;

    // 处理特殊的边界小规模情况
    if (x == 1 && y == 0) {
        printf("3\n");
    } else if (x == 2 && y == 2) {
        printf("4\n");
    } else {
        // 通用无限棋盘马步距离公式
        // k1 为基于单轴跨度的下界：由于每步在主轴最多走2格，所以至少需要 ceil(x/2) 步
        long long k1 = (x + 1) / 2;
        // k2 为基于曼哈顿距离和的下界：由于每步总曼哈顿位移为3，所以至少需要 ceil((x+y)/3) 步
        long long k2 = (x + y + 2) / 3;
        
        long long k = (k1 > k2) ? k1 : k2;

        // 根据马步移动规律，每走一步都会改变坐标和的奇偶性
        // 最终步数 k 必须与总位移 (x+y) 的奇偶性保持一致
        long long ans = k + (k + x + y) % 2;
        
        printf("%lld\n", ans);
    }

    return 0;
}

## I 直方图最大矩形

In [ ]:
#include <stdlib.h>

/**
 * 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
 *
 * 寻找直方图中能形成的最大矩形面积
 * 
 * @param heights int整型一维数组 
 * @param heightsLen int heights数组长度
 * @return int整型
 */
int largestRectangleArea(int* heights, int heightsLen ) {
    // 处理特殊情况：数组长度为 0
    if (heightsLen <= 0) {
        return 0;
    }

    // 分配栈空间，栈中存储的是数组的下标
    // 为了方便处理最后留在栈中的元素，我们多分配一个空间处理虚拟的“哨兵”
    int* stack = (int*)malloc(sizeof(int) * (heightsLen + 1));
    int top = -1; // 栈顶指针，-1 表示空栈
    long long maxArea = 0; // 使用 long long 防止中间计算溢出

    // 遍历 heights 数组，循环到 heightsLen 是为了处理最后留在栈中的元素（哨兵逻辑）
    for (int i = 0; i <= heightsLen; i++) {
        // 定义当前高度：如果到了 heightsLen，则视为高度为 0 的哨兵，确保清空栈
        int currentHeight = (i == heightsLen) ? 0 : heights[i];

        // 如果栈不为空，且当前高度小于栈顶高度，说明找到了栈顶柱子的右边界
        while (top != -1 && heights[stack[top]] >= currentHeight) {
            // 弹出栈顶元素，作为矩形的高度
            int h = heights[stack[top--]];
            
            // 计算矩形的宽度：
            // 如果栈弹空了，说明该高度 h 可以延伸到最左边，宽度即为 i
            // 如果栈不为空，左边界就是新的栈顶下标，宽度为 (当前下标 i - 新栈顶下标 - 1)
            int w = (top == -1) ? i : (i - stack[top] - 1);
            
            long long area = (long long)h * w;
            if (area > maxArea) {
                maxArea = area;
            }
        }

        // 将当前下标入栈
        stack[++top] = i;
    }

    // 释放动态分配的内存
    free(stack);

    return (int)maxArea;
}## add your code here

## J 消防局的设立

In [ ]:
#include <stdio.h>
#include <stdlib.h>

#define MAXN 200005  // 根据题目规模调整，通常 10^5 或 2*10^5
#define INF 10       // 只要大于 2 即可

// 链式前向星建图
int head[MAXN], to[MAXN * 2], nxt[MAXN * 2], edge_count = 0;

void add_edge(int u, int v) {
    to[++edge_count] = v;
    nxt[edge_count] = head[u];
    head[u] = edge_count;
}

int parent[MAXN], queue[MAXN], dist[MAXN];

int main() {
    int n;
    if (scanf("%d", &n) != 1) return 0;
    
    if (n <= 0) { printf("0\n"); return 0; }
    if (n == 1) { printf("1\n"); return 0; }

    // 读取 n-1 条边。根据示例，第 i 行表示第 i+1 个节点的连接情况
    for (int i = 2; i <= n; i++) {
        int p;
        if (scanf("%d", &p) == 1) {
            add_edge(i, p);
            add_edge(p, i);
        }
    }

    // BFS 确定节点层级和父子关系，根节点设为 1
    int l = 0, r = 0;
    queue[r++] = 1;
    parent[1] = 0;
    while (l < r) {
        int u = queue[l++];
        for (int i = head[u]; i; i = nxt[i]) {
            int v = to[i];
            if (v != parent[u]) {
                parent[v] = u;
                queue[r++] = v;
            }
        }
    }

    // 初始化所有节点到消防局的距离为无穷大
    for (int i = 1; i <= n; i++) dist[i] = INF;

    int station_count = 0;
    // 按照深度从大到小（逆 BFS 序）处理节点
    for (int i = n - 1; i >= 0; i--) {
        int u = queue[i];
        
        // 检查邻居节点是否已经有消防局能覆盖当前点
        for (int j = head[u]; j; j = nxt[j]) {
            if (dist[to[j]] + 1 < dist[u]) {
                dist[u] = dist[to[j]] + 1;
            }
        }

        // 如果当前节点 u 没有被覆盖（距离 > 2）
        if (dist[u] > 2) {
            station_count++;
            
            // 贪心选择：将消防局设在爷爷节点
            int s = u;
            if (parent[s]) s = parent[s];
            if (parent[s]) s = parent[s];
            
            // 更新新设立消防局周围距离为 2 以内的点
            dist[s] = 0;
            for (int j = head[s]; j; j = nxt[j]) {
                int v = to[j];
                if (dist[v] > 1) dist[v] = 1; // 距离为 1 的点
                for (int k = head[v]; k; k = nxt[k]) {
                    int w = to[k];
                    if (dist[w] > 2) dist[w] = 2; // 距离为 2 的点
                }
            }
        }
    }

    printf("%d\n", station_count);

    return 0;
}